# graphlift: GraphSAGE on YelpChi and Amazon (Google Colab)

Fills the two *pending* GraphSAGE cells in the Graph Lift Index.

1. **Runtime → Change runtime type → T4 GPU → Save.**
2. **Runtime → Run all.** If Colab warns that the notebook is not authored by Google, choose *Run anyway*.
3. When the last cell finishes, `graphlift_gpu_results.zip` downloads automatically.

About 15–25 minutes in total. Keep this tab open while it runs: free Colab
sessions stop after a while without activity.

In [ ]:
import torch
print("PyTorch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU, then Runtime > Run all again."

In [ ]:
# graphlift from GitHub (PyTorch is already installed on Colab)
!rm -rf graphlift && git clone -q --depth 1 https://github.com/RhysMiranda/graphlift
!pip install -q -e graphlift
import subprocess
v = subprocess.run(["python", "-c", "import graphlift; print(graphlift.__version__)"], capture_output=True, text=True).stdout.strip()
print("graphlift", v)
assert v >= "1.0.0", "GitHub still has an older graphlift. Push version 1.0.0 first, then Run all again."

In [ ]:
# YelpChi and Amazon, as released with CARE-GNN (about 45 MB)
!rm -rf CARE-GNN && git clone -q --depth 1 https://github.com/YingtongDou/CARE-GNN
!mkdir -p raw && unzip -o -q CARE-GNN/data/YelpChi.zip -d raw && unzip -o -q CARE-GNN/data/Amazon.zip -d raw
!python graphlift/examples/care/convert_care.py raw/Amazon.mat data/amazon
!python graphlift/examples/care/convert_care.py raw/YelpChi.mat data/yelpchi

In [ ]:
%%time
# Amazon: 3 seeds x 5 folds, four tree models plus GraphSAGE
!graphlift audit data/amazon --gnn --out data/amazon_report | grep -E "seed|Verdict|Lift|lift"

In [ ]:
%%time
# YelpChi: same, on the larger graph
!graphlift audit data/yelpchi --gnn --out data/yelpchi_report | grep -E "seed|Verdict|Lift|lift"

In [ ]:
# Zip the four report files and download them
!mkdir -p results_gpu
!cp data/amazon_report/report.md results_gpu/amazon.md && cp data/amazon_report/report.json results_gpu/amazon.json
!cp data/yelpchi_report/report.md results_gpu/yelpchi.md && cp data/yelpchi_report/report.json results_gpu/yelpchi.json
!rm -f graphlift_gpu_results.zip && zip -q -r graphlift_gpu_results.zip results_gpu
from google.colab import files
files.download("graphlift_gpu_results.zip")
print("Done. If the download did not start, open the Files panel on the left and download graphlift_gpu_results.zip.")